# 2.8 — Proximal & Subgradient Methods

Many useful ML objectives have corners: absolute value creates sparsity, hinge loss creates margins, and constraints create hard boundaries. This lesson shows how to optimize those nonsmooth pieces without pretending they are smooth: subgradients generalize slopes, proximal operators solve tiny penalty-aware correction problems, and proximal gradient combines ordinary gradients with exact nonsmooth updates.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build proximal and subgradient methods one idea at a time. Run each cell in order and inspect the printed values — every slope choice, threshold, and update is exposed so the nonsmooth logic is visible instead of hidden inside an optimizer. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized formulas, and numerical checks.
import matplotlib.pyplot as plt  # visual diagnostics for corners, updates, and losses.
np.random.seed(0)  # make all stochastic demos reproducible.

### 1. Subgradients: slopes that support a convex function

For a differentiable convex function, the tangent line sits below the graph everywhere. A **subgradient** keeps exactly that property even at a corner: a vector `g` is valid at `x` if `f(z) >= f(x) + g*(z-x)` for every `z`. For `f(x)=|x|`, the corner at zero has not one slope but a whole interval of supporting slopes.

In [ ]:
z_w = np.linspace(-3, 3, 301)  # points where we test support lines.
f_w = np.abs(z_w)  # nonsmooth convex function with a corner at 0.
x0_w = 0.0  # the corner where ordinary derivative is undefined.
g_choices_w = np.array([-1.0, -0.25, 0.5, 1.0])  # all are valid subgradients at zero.
print("candidate subgradients at 0:", g_choices_w)
print("function value at corner:", abs(x0_w))

▶ What you'll see: several different slopes are proposed for the same corner.

In [ ]:
for g_w in g_choices_w:
    support_w = abs(x0_w) + g_w * (z_w - x0_w)  # supporting line from the subgradient definition.
    gap_w = f_w - support_w  # must be nonnegative everywhere if g is valid.
    print(f"g={g_w: .2f}, smallest f-support gap = {gap_w.min():.3f}")
    assert gap_w.min() >= -1e-12

▶ What you'll see: every slope between -1 and 1 produces a line that never rises above `|z|`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(z_w, f_w, color="black", label="|z|")
for g_w in g_choices_w:
    plt.plot(z_w, g_w * z_w, linestyle="--", label=f"support g={g_w:g}")
plt.title("1: many valid subgradients at a corner")
plt.xlabel("z"); plt.ylabel("value"); plt.legend(); plt.show()

▶ What you'll see: the dashed lines touch the V-shaped absolute value at zero and stay underneath it.

*Why it's done this way:* the supporting-line definition preserves the global geometry that makes convex optimization reliable. At a kink, there is no unique tangent slope, so the algorithm is allowed to pick any slope whose line certifies that the function lies above it everywhere.

### 2. Subgradient descent: keep the template, change the slope

Subgradient descent uses the familiar update `x <- x - eta*g`, but `g` may be any valid subgradient. Unlike smooth gradients, a subgradient direction is not guaranteed to decrease the objective every single step, so we often use diminishing step sizes to make the path settle down.

In [ ]:
def abs_subgrad_w(x_w):
    if x_w > 0:
        return 1.0
    if x_w < 0:
        return -1.0
    return 0.0  # a deliberate valid choice from [-1, 1] at the corner.

x_w = 2.75  # start to the right of the minimum.
eta0_w = 0.8  # initial step size.
path_w = [x_w]
print("start x:", x_w, "f(x):", abs(x_w))

▶ What you'll see: the iterate starts away from the nonsmooth minimum at zero.

In [ ]:
for t_w in range(1, 9):
    eta_w = eta0_w / np.sqrt(t_w)  # diminishing step: large early, smaller later.
    g_w = abs_subgrad_w(x_w)  # valid generalized slope at current x.
    x_w = x_w - eta_w * g_w  # subgradient descent update.
    path_w.append(x_w)
    print(f"t={t_w}, eta={eta_w:.3f}, g={g_w:+.0f}, x={x_w:.3f}, |x|={abs(x_w):.3f}")
assert abs(path_w[-1]) < abs(path_w[0])

▶ What you'll see: the iterate moves toward zero, may cross it, then takes smaller corrective steps.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(path_w, marker="o", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("2: subgradient iterates for |x|")
plt.xlabel("iteration"); plt.ylabel("x"); plt.show()

▶ What you'll see: diminishing steps damp the zig-zag near the kink.

*Why it's done this way:* the sign of `x` is the only slope information available away from the corner, and it still points toward lower absolute value. Diminishing steps matter because the slope magnitude does not shrink as `x` approaches zero, so a fixed step can bounce forever.

### 3. Proximal operators: optimize the kink exactly nearby

A proximal operator asks a small optimization question: after proposing a point `v`, what `x` best balances staying close to `v` with paying a nonsmooth penalty `lambda*r(x)`? For `r(x)=|x|`, the prox solves `min_x 0.5*(x-v)^2 + lambda*|x|`.

In [ ]:
v_grid_w = np.linspace(-3, 3, 601)  # candidate x values for brute-force checking.
v0_w = 2.0  # point we want to denoise/shrink.
lam_w = 1.0  # penalty strength.
obj_w = 0.5 * (v_grid_w - v0_w) ** 2 + lam_w * np.abs(v_grid_w)
best_x_w = v_grid_w[np.argmin(obj_w)]
print("v:", v0_w, "lambda:", lam_w, "brute prox:", best_x_w)
assert abs(best_x_w - 1.0) < 1e-12

▶ What you'll see: the best point is 1.0, not 2.0, because paying absolute-value penalty encourages shrinkage.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(v_grid_w, obj_w, color="purple")
plt.axvline(v0_w, color="gray", linestyle="--", label="v")
plt.axvline(best_x_w, color="seagreen", linestyle="--", label="prox")
plt.title("3: prox solves a local penalty tradeoff")
plt.xlabel("candidate x"); plt.ylabel("0.5(x-v)^2 + λ|x|"); plt.legend(); plt.show()

▶ What you'll see: the minimum of the prox objective lies between zero and `v`.

In [ ]:
v_vec_w = np.array([-2.0, -0.5, 0.5, 2.0])
soft_w = np.sign(v_vec_w) * np.maximum(np.abs(v_vec_w) - lam_w, 0.0)
print("v:", v_vec_w)
print("soft-threshold prox:", soft_w)
assert np.allclose(soft_w, [-1.0, 0.0, 0.0, 1.0])

▶ What you'll see: large values shrink by 1, while small values snap exactly to zero.

*Why it's done this way:* the quadratic term says “do not move too far from `v`,” while `|x|` says “prefer small or zero values.” Solving that one-dimensional convex problem exactly gives soft-thresholding, which preserves the corner instead of smoothing it away.

### 4. Proximal gradient: smooth step first, prox correction second

Composite objectives split naturally into a smooth part plus a nonsmooth penalty, for example `0.5||Aw-y||^2 + lambda||w||_1`. Proximal gradient first takes an ordinary gradient step on the smooth error, then applies the proximal operator of the nonsmooth penalty. The threshold is `eta*lambda`, not just `lambda`.

In [ ]:
A_w = np.eye(2)
y_w = np.array([-1.0, 0.35])
w_w = np.array([2.0, -0.25])
eta_w = 0.25
lam_w = 0.8
resid_w = A_w @ w_w - y_w
grad_w = A_w.T @ resid_w  # gradient of 0.5||Aw-y||^2.
print("residual:", np.round(resid_w, 3))
print("smooth gradient:", np.round(grad_w, 3))

▶ What you'll see: the residual and gradient come only from the differentiable least-squares part.

In [ ]:
v_w = w_w - eta_w * grad_w  # gradient step before applying the L1 penalty.
threshold_w = eta_w * lam_w
w_next_w = np.sign(v_w) * np.maximum(np.abs(v_w) - threshold_w, 0.0)
print("gradient step v:", np.round(v_w, 3))
print("threshold eta*lambda:", threshold_w)
print("prox-gradient w_next:", np.round(w_next_w, 3))
assert np.allclose(w_next_w, [1.05, 0.0])

▶ What you'll see: the second coordinate becomes exactly zero after the prox correction.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["w0 before", "w1 before", "w0 after", "w1 after"], [w_w[0], w_w[1], w_next_w[0], w_next_w[1]], color=["gray", "gray", "teal", "teal"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("4: proximal gradient creates sparsity")
plt.xticks(rotation=20); plt.ylabel("coefficient"); plt.show()

▶ What you'll see: proximal gradient both follows the data gradient and performs exact L1 shrinkage.

*Why it's done this way:* the smooth term has a trustworthy gradient, while the L1 term has a corner that should be handled exactly. Splitting the update lets each piece use the mathematical tool that fits it best.

### 5. Hinge loss: subgradients for margins

The hinge loss `max(0, 1 - margin)` is zero once the classification margin is at least one. Before that, its subgradient keeps pushing the model; after that, the example stops pulling. This is the nonsmooth mechanism behind margin-based learning.

In [ ]:
margins_w = np.array([-0.5, 0.25, 1.0, 1.5, 2.0])
loss_w = np.maximum(0.0, 1.0 - margins_w)
subgrad_m_w = np.where(margins_w < 1.0, -1.0, 0.0)  # choose 0 at the kink margin=1.
print("margins:", margins_w)
print("hinge losses:", loss_w)
print("chosen slopes wrt margin:", subgrad_m_w)
assert np.allclose(loss_w, [1.5, 0.75, 0.0, 0.0, 0.0])

▶ What you'll see: only margin-violating examples have positive loss and nonzero slope.

In [ ]:
x_w = np.array([2.0, -1.0])
y_w = 1.0
w_w = np.array([0.2, 0.1])
margin_w = y_w * float(w_w @ x_w)
if margin_w < 1.0:
    grad_hinge_w = -y_w * x_w
else:
    grad_hinge_w = np.zeros_like(x_w)
print("margin:", round(margin_w, 3))
print("subgradient wrt w:", grad_hinge_w)
assert np.allclose(grad_hinge_w, [-2.0, 1.0])

▶ What you'll see: a violated margin produces a subgradient that pushes `w` toward the labeled example.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(margins_w, loss_w, marker="o", color="crimson")
plt.axvline(1.0, color="black", linestyle="--", label="margin satisfied")
plt.title("5: hinge loss goes flat after margin 1")
plt.xlabel("margin y wᵀx"); plt.ylabel("max(0, 1-margin)"); plt.legend(); plt.show()

▶ What you'll see: the loss is a straight declining line until margin 1, then a flat zero plateau.

*Why it's done this way:* the hinge corner encodes the modeling rule “good enough margin is enough.” Subgradients let the optimizer respect that exact flat region instead of wasting effort increasing already-safe margins.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Plot a nonsmooth absolute value

**Goal.** Build the simplest convex kink, because every later method is about optimizing useful objectives where ordinary derivatives can fail.

In [ ]:
x_b1 = np.linspace(-3, 3, 121)
y_b1 = np.abs(x_b1)
print("x shape:", x_b1.shape, "min |x|:", y_b1.min())
assert y_b1.min() == 0.0
plt.figure(figsize=(4, 3))
plt.plot(x_b1, y_b1, color="black")
plt.title("Basic 1: |x| has a corner")
plt.xlabel("x"); plt.ylabel("|x|"); plt.show()

▶ What you'll see: a V-shaped graph with a sharp corner at zero.

👀 Takeaway: nonsmooth does not mean broken; it means the usual single derivative may not exist at key points.

### Basic 2 — Test a supporting line

**Goal.** Verify the subgradient inequality for one slope at the absolute-value corner.

In [ ]:
z_b2 = np.array([-2.0, 0.0, 2.0])
g_b2 = 0.5
support_b2 = g_b2 * z_b2
truth_b2 = np.abs(z_b2)
print("support:", support_b2)
print("|z|:", truth_b2)
assert np.all(truth_b2 >= support_b2)

▶ What you'll see: the candidate line is below `|z|` at all checked points.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(z_b2, truth_b2, marker="o", color="black", label="|z|")
plt.plot(z_b2, support_b2, marker="o", linestyle="--", color="teal", label=f"support g={g_b2}")
plt.title("Basic 2: support line stays below |z|")
plt.xlabel("z"); plt.ylabel("value"); plt.legend(); plt.show()

▶ What you'll see: the dashed supporting line touches the corner direction and remains no higher than the absolute-value points.

👀 Takeaway: a subgradient is valid because its line supports the convex function from below.

### Basic 3 — See the whole subgradient interval

**Goal.** Check which slopes are valid at `x=0` for `|x|`, because the corner has many legal generalized slopes.

In [ ]:
slopes_b3 = np.array([-1.5, -1.0, 0.0, 0.75, 1.0, 1.5])
z_b3 = np.linspace(-2, 2, 401)
valid_b3 = []
for g_b3 in slopes_b3:
    valid_b3.append(bool(np.all(np.abs(z_b3) >= g_b3 * z_b3 - 1e-12)))
print("slopes:", slopes_b3)
print("valid at zero:", valid_b3)
assert valid_b3 == [False, True, True, True, True, False]

▶ What you'll see: only slopes in `[-1, 1]` pass the support-line test.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.scatter(slopes_b3, np.zeros_like(slopes_b3), c=valid_b3, cmap="coolwarm", s=80)
plt.axvspan(-1, 1, color="seagreen", alpha=0.15, label="valid subgradient set")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 3: subgradient interval at zero")
plt.xlabel("candidate slope g"); plt.yticks([]); plt.legend(); plt.show()

▶ What you'll see: valid candidates lie inside the highlighted interval from -1 to 1.

👀 Takeaway: subgradient choices at a kink form a set, not a single number.

### Basic 4 — Take one subgradient step

**Goal.** Perform one update on `|x|`, because subgradient descent keeps the gradient-descent template.

In [ ]:
x_b4 = 2.0
eta_b4 = 0.3
g_b4 = 1.0 if x_b4 > 0 else -1.0 if x_b4 < 0 else 0.0
x_next_b4 = x_b4 - eta_b4 * g_b4
print("x before:", x_b4, "g:", g_b4, "x after:", x_next_b4)
assert round(x_next_b4, 3) == 1.7

▶ What you'll see: a positive point moves left toward the minimum at zero.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot([0, 1], [x_b4, x_next_b4], marker="o", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 4: one subgradient step toward zero")
plt.xlabel("step"); plt.ylabel("x"); plt.xticks([0, 1], ["before", "after"]); plt.show()

▶ What you'll see: the update lowers the positive iterate by exactly the step size.

👀 Takeaway: away from a kink, subgradient descent often looks exactly like ordinary gradient descent.

### Basic 5 — Compare fixed and diminishing steps

**Goal.** Show why nonsmooth methods often shrink the step size near a corner.

In [ ]:
steps_b5 = np.arange(1, 9)
fixed_b5 = np.full_like(steps_b5, 0.6, dtype=float)
diminish_b5 = 0.6 / np.sqrt(steps_b5)
print("fixed:", fixed_b5[:4])
print("diminishing:", np.round(diminish_b5[:4], 3))
plt.figure(figsize=(4, 3))
plt.plot(steps_b5, fixed_b5, marker="o", label="fixed")
plt.plot(steps_b5, diminish_b5, marker="o", label="diminishing")
plt.title("Basic 5: step-size schedules")
plt.xlabel("iteration"); plt.ylabel("eta"); plt.legend(); plt.show()

▶ What you'll see: the diminishing schedule starts equal but gradually becomes more cautious.

👀 Takeaway: shrinking steps reduce persistent bouncing when the subgradient magnitude does not vanish near the optimum.

### Basic 6 — Brute-force a scalar prox

**Goal.** Solve the proximal problem by search once, because the definition is an optimization problem before it becomes a formula.

In [ ]:
v_b6 = -2.0
lam_b6 = 1.0
candidates_b6 = np.linspace(-3, 1, 401)
objective_b6 = 0.5 * (candidates_b6 - v_b6) ** 2 + lam_b6 * np.abs(candidates_b6)
best_b6 = candidates_b6[np.argmin(objective_b6)]
print("best x:", round(float(best_b6), 3))
assert abs(best_b6 + 1.0) < 1e-12

▶ What you'll see: the prox of `-2` with threshold `1` is `-1`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.plot(candidates_b6, objective_b6, color="purple")
plt.axvline(v_b6, color="gray", linestyle="--", label="v")
plt.axvline(best_b6, color="seagreen", linestyle="--", label="prox")
plt.title("Basic 6: scalar prox objective")
plt.xlabel("candidate x"); plt.ylabel("0.5(x-v)^2 + λ|x|"); plt.legend(); plt.show()

▶ What you'll see: the minimum shifts from the input toward zero because the absolute-value penalty charges magnitude.

👀 Takeaway: prox balances staying near `v` against paying the nonsmooth penalty.

### Basic 7 — Apply soft-thresholding

**Goal.** Use the closed-form prox of `lambda*|x|`, because it is the core update behind L1 sparsity.

In [ ]:
v_b7 = np.array([-2.0, -0.5, 0.5, 2.0])
lam_b7 = 1.0
prox_b7 = np.sign(v_b7) * np.maximum(np.abs(v_b7) - lam_b7, 0.0)
print("input:", v_b7)
print("prox:", prox_b7)
assert np.allclose(prox_b7, [-1.0, 0.0, 0.0, 1.0])

▶ What you'll see: small magnitudes become exactly zero, while large magnitudes shrink toward zero.

In [ ]:
grid_b7 = np.linspace(-3, 3, 301)
soft_curve_b7 = np.sign(grid_b7) * np.maximum(np.abs(grid_b7) - lam_b7, 0.0)
plt.figure(figsize=(4.5, 3))
plt.plot(grid_b7, soft_curve_b7, color="seagreen", label="soft-threshold")
plt.scatter(v_b7, prox_b7, color="black", zorder=3, label="examples")
plt.axhline(0, color="black", linewidth=0.8); plt.axvline(0, color="black", linewidth=0.8)
plt.title("Basic 7: soft-thresholding curve")
plt.xlabel("input v"); plt.ylabel("prox(v)"); plt.legend(); plt.show()

▶ What you'll see: the curve has a flat zero region for inputs whose magnitude is below the threshold.

👀 Takeaway: soft-thresholding is shrinkage plus exact sparsification.

### Basic 8 — Remember the prox-gradient threshold scale

**Goal.** Compute `eta*lambda`, because proximal gradient uses the step size inside the prox penalty.

In [ ]:
eta_b8 = 0.2
lam_b8 = 0.5
threshold_b8 = eta_b8 * lam_b8
v_b8 = np.array([0.08, 0.3, -0.6])
prox_b8 = np.sign(v_b8) * np.maximum(np.abs(v_b8) - threshold_b8, 0.0)
print("threshold:", threshold_b8)
print("prox:", np.round(prox_b8, 3))
assert np.allclose(prox_b8, [0.0, 0.2, -0.5])

▶ What you'll see: the threshold is `0.1`, not `0.5`.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(v_b8)) - 0.18, v_b8, width=0.36, label="v", color="gray")
plt.bar(np.arange(len(v_b8)) + 0.18, prox_b8, width=0.36, label="prox", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 8: threshold scale eta times lambda")
plt.xlabel("coordinate"); plt.ylabel("value"); plt.legend(); plt.show()

▶ What you'll see: only the coordinate inside the ±0.1 threshold snaps to zero.

👀 Takeaway: forgetting the learning-rate scale makes proximal gradient shrink far too aggressively.

### Basic 9 — Compute hinge losses

**Goal.** Evaluate hinge loss for several margins, because SVM-style objectives are nonsmooth at the margin boundary.

In [ ]:
margins_b9 = np.array([0.25, 1.0, 1.5])
losses_b9 = np.maximum(0.0, 1.0 - margins_b9)
print("margins:", margins_b9)
print("hinge losses:", losses_b9)
assert np.allclose(losses_b9, [0.75, 0.0, 0.0])

▶ What you'll see: loss disappears once the margin reaches one.

In [ ]:
margin_grid_b9 = np.linspace(-0.5, 2.0, 200)
hinge_curve_b9 = np.maximum(0.0, 1.0 - margin_grid_b9)
plt.figure(figsize=(4.5, 3))
plt.plot(margin_grid_b9, hinge_curve_b9, color="crimson")
plt.scatter(margins_b9, losses_b9, color="black", zorder=3)
plt.axvline(1.0, color="black", linestyle="--", label="margin 1")
plt.title("Basic 9: hinge loss kink")
plt.xlabel("margin"); plt.ylabel("hinge loss"); plt.legend(); plt.show()

▶ What you'll see: the piecewise-linear loss hits zero at margin one and stays flat afterward.

👀 Takeaway: hinge loss pushes only examples that violate or touch the desired margin.

### Basic 10 — Choose a hinge subgradient

**Goal.** Convert a margin violation into a parameter update direction.

In [ ]:
x_b10 = np.array([2.0, -1.0])
y_b10 = 1.0
w_b10 = np.array([0.2, 0.1])
margin_b10 = y_b10 * float(w_b10 @ x_b10)
grad_b10 = -y_b10 * x_b10 if margin_b10 < 1.0 else np.zeros_like(x_b10)
print("margin:", round(margin_b10, 3))
print("hinge subgradient wrt w:", grad_b10)
assert np.allclose(grad_b10, [-2.0, 1.0])

▶ What you'll see: the violated example produces a nonzero vector that would raise the margin after descent.

In [ ]:
plt.figure(figsize=(4, 3))
plt.arrow(0, 0, w_b10[0], w_b10[1], color="gray", head_width=0.05, length_includes_head=True, label="current w")
plt.arrow(w_b10[0], w_b10[1], -0.08 * grad_b10[0], -0.08 * grad_b10[1], color="crimson", head_width=0.05, length_includes_head=True, label="descent direction")
plt.scatter([x_b10[0]], [x_b10[1]], color="teal", s=80, label="example x")
plt.axhline(0, color="black", linewidth=0.8); plt.axvline(0, color="black", linewidth=0.8)
plt.title("Basic 10: hinge subgradient direction")
plt.xlabel("w0 / x0"); plt.ylabel("w1 / x1"); plt.legend(); plt.show()

▶ What you'll see: the descent arrow points in the direction that increases alignment with the positive example.

👀 Takeaway: hinge subgradients turn margin violations into direct corrective forces.

## 🟡 Easy

### Easy 1 — Run subgradient descent on absolute value

**Goal.** Iterate subgradient descent with a diminishing schedule, because nonsmooth objectives often need cautious convergence behavior.

In [ ]:
x_e1 = 3.0
path_e1 = [x_e1]
for t_e1 in range(1, 16):
    eta_e1 = 0.7 / np.sqrt(t_e1)
    g_e1 = 1.0 if x_e1 > 0 else -1.0 if x_e1 < 0 else 0.0
    x_e1 = x_e1 - eta_e1 * g_e1
    path_e1.append(x_e1)
print("final x:", round(x_e1, 3), "final |x|:", round(abs(x_e1), 3))
assert abs(x_e1) < 0.5
plt.figure(figsize=(5, 3))
plt.plot(path_e1, marker="o", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 1: subgradient path")
plt.xlabel("iteration"); plt.ylabel("x"); plt.show()

▶ What you'll see: the path approaches zero and then oscillates with shrinking amplitude.

👀 Takeaway: subgradient methods may not decrease every step, but a sensible schedule can still drive them toward the optimum.

### Easy 2 — Compare gradient shrinkage and prox shrinkage

**Goal.** Contrast ordinary L2-style shrinkage with L1 soft-thresholding, because only the prox step creates exact zeros.

In [ ]:
w_e2 = np.array([-0.3, 0.2, 1.5])
eta_e2 = 0.4
lam_e2 = 0.5
l2_like_e2 = w_e2 - eta_e2 * lam_e2 * w_e2
l1_prox_e2 = np.sign(w_e2) * np.maximum(np.abs(w_e2) - eta_e2 * lam_e2, 0.0)
print("L2-like shrink:", np.round(l2_like_e2, 3))
print("L1 prox shrink:", np.round(l1_prox_e2, 3))
assert l1_prox_e2[1] == 0.0
plt.figure(figsize=(5, 3))
plt.bar(np.arange(3) - 0.18, l2_like_e2, width=0.36, label="smooth shrink")
plt.bar(np.arange(3) + 0.18, l1_prox_e2, width=0.36, label="L1 prox")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 2: shrink versus threshold")
plt.legend(); plt.show()

▶ What you'll see: smooth shrinkage reduces magnitudes, while L1 prox can set a coordinate exactly to zero.

👀 Takeaway: sparsity comes from the nonsmooth corner, not from generic shrinkage alone.

### Easy 3 — One proximal-gradient step for Lasso

**Goal.** Combine a least-squares gradient step with an L1 prox correction.

In [ ]:
A_e3 = np.eye(2)
y_e3 = np.array([-1.0, 0.35])
w_e3 = np.array([2.0, -0.25])
eta_e3 = 0.25
lam_e3 = 0.8
grad_e3 = A_e3.T @ (A_e3 @ w_e3 - y_e3)
v_e3 = w_e3 - eta_e3 * grad_e3
w_next_e3 = np.sign(v_e3) * np.maximum(np.abs(v_e3) - eta_e3 * lam_e3, 0.0)
print("gradient:", np.round(grad_e3, 3))
print("v:", np.round(v_e3, 3), "w_next:", np.round(w_next_e3, 3))
assert np.allclose(w_next_e3, [1.05, 0.0])

▶ What you'll see: the smooth step proposes `v`, then the L1 prox zeros the small coordinate.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(np.arange(len(w_e3)) - 0.25, w_e3, width=0.25, label="w", color="gray")
plt.bar(np.arange(len(v_e3)), v_e3, width=0.25, label="gradient step v", color="orange")
plt.bar(np.arange(len(w_next_e3)) + 0.25, w_next_e3, width=0.25, label="prox result", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Easy 3: gradient step then L1 prox")
plt.xlabel("coordinate"); plt.ylabel("value"); plt.legend(); plt.show()

▶ What you'll see: the prox correction shrinks both coordinates and sends the small second coordinate to zero.

👀 Takeaway: proximal gradient is “differentiate what is smooth, prox what is nonsmooth.”

### Easy 4 — Train a tiny L1-regularized linear model

**Goal.** Repeat proximal-gradient steps, because L1 penalties can recover sparse coefficients over iterations.

In [ ]:
A_e4 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, 0.0]])
y_e4 = np.array([1.0, 0.0, 1.0, 2.0])
w_e4 = np.array([0.0, 0.8])
eta_e4 = 0.1
lam_e4 = 0.6
losses_e4 = []
for t_e4 in range(60):
    grad_e4 = A_e4.T @ (A_e4 @ w_e4 - y_e4)
    v_e4 = w_e4 - eta_e4 * grad_e4
    w_e4 = np.sign(v_e4) * np.maximum(np.abs(v_e4) - eta_e4 * lam_e4, 0.0)
    losses_e4.append(0.5 * np.sum((A_e4 @ w_e4 - y_e4) ** 2) + lam_e4 * np.sum(np.abs(w_e4)))
print("final w:", np.round(w_e4, 3))
assert abs(w_e4[1]) < 1e-8
plt.figure(figsize=(5, 3))
plt.plot(losses_e4, color="purple")
plt.title("Easy 4: proximal-gradient objective")
plt.xlabel("iteration"); plt.ylabel("objective"); plt.show()

▶ What you'll see: the objective falls and the irrelevant second coefficient becomes zero.

👀 Takeaway: repeated prox steps make sparsity a stable optimization outcome.

### Easy 5 — Update a linear classifier with hinge loss

**Goal.** Apply hinge subgradients over a tiny dataset, because each example should pull only when its margin is too small.

In [ ]:
X_e5 = np.array([[2.0, 1.0], [1.0, -1.0], [-1.0, -1.0], [-2.0, 1.0]])
y_e5 = np.array([1.0, 1.0, -1.0, -1.0])
w_e5 = np.zeros(2)
eta_e5 = 0.2
for i_e5 in range(len(y_e5)):
    margin_e5 = y_e5[i_e5] * float(w_e5 @ X_e5[i_e5])
    grad_e5 = -y_e5[i_e5] * X_e5[i_e5] if margin_e5 < 1.0 else np.zeros(2)
    w_e5 = w_e5 - eta_e5 * grad_e5
    print(f"i={i_e5}, margin={margin_e5:.3f}, w={np.round(w_e5, 3)}")
assert w_e5[0] > 0
plt.figure(figsize=(4, 3))
plt.scatter(X_e5[:, 0], X_e5[:, 1], c=y_e5, cmap="coolwarm", s=80)
plt.arrow(0, 0, w_e5[0], w_e5[1], color="black", head_width=0.08)
plt.title("Easy 5: hinge updates build a separator")
plt.xlabel("x0"); plt.ylabel("x1"); plt.show()

▶ What you'll see: violated examples rotate the weight vector toward a separating direction.

👀 Takeaway: once an example has enough margin, hinge loss stops spending updates on it.

## 🔴 Advanced

### Advanced 1 — Compare fixed and diminishing subgradient paths

**Goal.** Demonstrate the step-size pitfall, because a fixed step can bounce around a nonsmooth optimum forever.

In [ ]:
x0_a1 = 1.0
iters_a1 = 30
paths_a1 = []
for mode_a1 in ["fixed", "diminishing"]:
    x_a1 = x0_a1
    path_a1 = [x_a1]
    for t_a1 in range(1, iters_a1 + 1):
        eta_a1 = 0.3 if mode_a1 == "fixed" else 0.3 / np.sqrt(t_a1)
        g_a1 = 1.0 if x_a1 > 0 else -1.0 if x_a1 < 0 else 0.0
        x_a1 = x_a1 - eta_a1 * g_a1
        path_a1.append(x_a1)
    paths_a1.append(path_a1)
print("fixed final abs:", round(abs(paths_a1[0][-1]), 3))
print("diminishing final abs:", round(abs(paths_a1[1][-1]), 3))
plt.figure(figsize=(5, 3))
plt.plot(paths_a1[0], label="fixed eta")
plt.plot(paths_a1[1], label="diminishing eta")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 1: subgradient step-size behavior")
plt.legend(); plt.show()

▶ What you'll see: the fixed path keeps bouncing with the same jump size, while the diminishing path damps down.

👀 Takeaway: nonsmooth optimization often needs step schedules that account for non-vanishing subgradient magnitudes.

### Advanced 2 — Trace a regularization path for soft-thresholding

**Goal.** Sweep `lambda`, because larger L1 penalties create more zeros and stronger shrinkage.

In [ ]:
v_a2 = np.array([-1.5, -0.4, 0.2, 2.0])
lambdas_a2 = np.linspace(0.0, 2.0, 9)
zeros_a2 = []
solutions_a2 = []
for lam_a2 in lambdas_a2:
    sol_a2 = np.sign(v_a2) * np.maximum(np.abs(v_a2) - lam_a2, 0.0)
    solutions_a2.append(sol_a2)
    zeros_a2.append(int(np.sum(sol_a2 == 0.0)))
print("zero counts:", zeros_a2)
assert zeros_a2[0] == 0 and zeros_a2[-1] == 4
plt.figure(figsize=(5, 3))
plt.plot(lambdas_a2, np.array(solutions_a2), marker="o")
plt.title("Advanced 2: L1 prox regularization path")
plt.xlabel("lambda"); plt.ylabel("prox coordinate value"); plt.show()

▶ What you'll see: coordinates hit zero at different λ values based on their initial magnitudes.

👀 Takeaway: L1 thresholds create sparse solutions progressively as the penalty grows.

### Advanced 3 — Solve a constrained prox by projection

**Goal.** Treat a constraint as an indicator penalty, because its proximal operator is projection onto the feasible set.

In [ ]:
v_a3 = np.array([1.2, -0.7, 0.4, 2.0])
lo_a3, hi_a3 = 0.0, 1.0
proj_a3 = np.clip(v_a3, lo_a3, hi_a3)
dist_before_a3 = np.linalg.norm(v_a3 - 0.5)
dist_after_a3 = np.linalg.norm(proj_a3 - 0.5)
print("v:", v_a3)
print("projection to [0,1]:", proj_a3)
assert np.all((proj_a3 >= lo_a3) & (proj_a3 <= hi_a3))
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(v_a3)) - 0.18, v_a3, width=0.36, label="v")
plt.bar(np.arange(len(v_a3)) + 0.18, proj_a3, width=0.36, label="projection")
plt.axhline(0, color="black", linewidth=0.8); plt.axhline(1, color="black", linewidth=0.8)
plt.title("Advanced 3: prox of a box constraint")
plt.legend(); plt.show()

▶ What you'll see: values outside `[0,1]` are clipped to the nearest feasible boundary.

👀 Takeaway: constrained optimization fits the proximal viewpoint by making infeasible points infinitely expensive.

### Advanced 4 — Robust regression with absolute loss subgradients

**Goal.** Fit a scalar median-like model under absolute loss, because subgradients are useful for robust objectives with kinks at zero residual.

In [ ]:
y_a4 = np.array([1.0, 1.2, 0.9, 1.1, 8.0])
theta_a4 = 0.0
path_a4 = []
for t_a4 in range(1, 80):
    residuals_a4 = theta_a4 - y_a4
    g_a4 = np.sum(np.sign(residuals_a4))  # subgradient of sum |theta-y_i|.
    eta_a4 = 0.08 / np.sqrt(t_a4)
    theta_a4 = theta_a4 - eta_a4 * g_a4
    path_a4.append(theta_a4)
print("robust theta:", round(theta_a4, 3), "mean:", round(float(np.mean(y_a4)), 3), "median:", round(float(np.median(y_a4)), 3))
assert abs(theta_a4 - np.median(y_a4)) < 0.3
plt.figure(figsize=(5, 3))
plt.plot(path_a4, color="teal")
plt.axhline(np.mean(y_a4), color="red", linestyle="--", label="mean")
plt.axhline(np.median(y_a4), color="black", linestyle="--", label="median")
plt.title("Advanced 4: absolute-loss subgradient")
plt.legend(); plt.show()

▶ What you'll see: the absolute-loss estimate moves near the median, not the outlier-pulled mean.

👀 Takeaway: nonsmooth absolute loss is robust because its subgradient depends on signs, not squared residual sizes.

### Advanced 5 — Monitor composite objective during proximal gradient

**Goal.** Verify that proximal-gradient updates reduce a Lasso-style objective when the step size is reasonable.

In [ ]:
A_a5 = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0], [2.0, 0.0, 0.0]])
y_a5 = np.array([1.0, 0.0, 1.0, 2.0])
w_a5 = np.array([0.0, 0.5, -0.5])
eta_a5 = 0.08
lam_a5 = 0.4
obj_a5 = []
for t_a5 in range(100):
    grad_a5 = A_a5.T @ (A_a5 @ w_a5 - y_a5)
    v_a5 = w_a5 - eta_a5 * grad_a5
    w_a5 = np.sign(v_a5) * np.maximum(np.abs(v_a5) - eta_a5 * lam_a5, 0.0)
    obj_a5.append(0.5 * np.sum((A_a5 @ w_a5 - y_a5) ** 2) + lam_a5 * np.sum(np.abs(w_a5)))
print("objective start -> end:", round(obj_a5[0], 3), "->", round(obj_a5[-1], 3))
print("final w:", np.round(w_a5, 3))
assert obj_a5[-1] < obj_a5[0]
plt.figure(figsize=(5, 3))
plt.plot(obj_a5, color="purple")
plt.title("Advanced 5: composite objective under prox-gradient")
plt.xlabel("iteration"); plt.ylabel("least squares + λ||w||₁"); plt.show()

▶ What you'll see: the composite objective decreases and at least one coordinate is strongly shrunk.

👀 Takeaway: proximal gradient is practical because it gives a simple monitored loop for smooth-plus-nonsmooth objectives.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Corners are not defects; proximal and subgradient methods are the calculus of useful nonsmooth objectives.

Convex objectives may have corners while retaining global structure. SVM hinge loss, Lasso, robust losses, and constrained composite objectives all use subgradients or proximal steps. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 271828
rng = np.random.default_rng(SEED)


def sigmoid(z):
    clipped = np.clip(z, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-clipped))


def soft_threshold(v, threshold):
    return np.sign(v) * np.maximum(np.abs(v) - threshold, 0.0)


def quadratic_loss(A, b, x):
    return 0.5 * float(x @ A @ x) - float(b @ x)


def quadratic_grad(A, b, x):
    return A @ x - b


def rosenbrock_loss(x):
    a = 1.0 - x[0]
    b = x[1] - x[0] ** 2
    ripple = 0.08 * np.sin(3.0 * x[0]) * np.cos(2.0 * x[1])
    return a ** 2 + 35.0 * b ** 2 + ripple


def rosenbrock_grad(x):
    dx = -2.0 * (1.0 - x[0]) - 140.0 * x[0] * (x[1] - x[0] ** 2)
    dy = 70.0 * (x[1] - x[0] ** 2)
    dx = dx + 0.24 * np.cos(3.0 * x[0]) * np.cos(2.0 * x[1])
    dy = dy - 0.16 * np.sin(3.0 * x[0]) * np.sin(2.0 * x[1])
    return np.array([dx, dy])


def make_logistic_data(n=96, d=2, seed=11):
    local = np.random.default_rng(seed)
    half = n // 2
    pos = local.normal(loc=1.15, scale=0.55, size=(half, d))
    neg = local.normal(loc=-1.05, scale=0.65, size=(n - half, d))
    X = np.vstack([pos, neg])
    y = np.hstack([np.ones(half), -np.ones(n - half)])
    return X, y


def make_sparse_logistic_data(n=140, d=32, seed=23):
    local = np.random.default_rng(seed)
    X = local.normal(size=(n, d))
    mask = local.random(size=X.shape) < 0.72
    X[mask] = 0.0
    true_w = np.zeros(d)
    true_w[:5] = np.array([1.4, -1.1, 0.9, -0.7, 0.45])
    logits = X @ true_w + 0.15 * local.normal(size=n)
    y = np.where(logits >= 0.0, 1.0, -1.0)
    return X, y


def logistic_loss(w, X, y, lam=0.0):
    margins = y * (X @ w)
    data = np.logaddexp(0.0, -margins).mean()
    penalty = 0.5 * lam * float(w @ w)
    return float(data + penalty)


def logistic_grad(w, X, y, lam=0.0):
    margins = y * (X @ w)
    weights = -y * sigmoid(-margins)
    grad = X.T @ weights / X.shape[0]
    return grad + lam * w


def l1_logistic_objective(w, X, y, lam=0.04):
    return logistic_loss(w, X, y, 0.0) + lam * float(np.abs(w).sum())


def project_box(x, lo=-2.0, hi=2.0):
    return np.clip(x, lo, hi)


def project_l2_ball(x, radius=2.0):
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x.copy()
    return x * (radius / norm)


def make_loss_surface_ladder():
    X2, y2 = make_logistic_data()
    Xh, yh = make_sparse_logistic_data()
    d = Xh.shape[1]
    A1 = np.array([[4.0, 1.0], [1.0, 3.0]])
    b1 = np.array([1.0, 2.0])
    A2 = np.array([[45.0, 18.0], [18.0, 9.0]])
    b2 = np.array([1.0, 0.4])
    return [
        {
            "id": "D1",
            "name": "quadratic bowl",
            "x0": np.array([0.0, 0.0]),
            "loss": lambda x, A=A1, b=b1: quadratic_loss(A, b, x),
            "grad": lambda x, A=A1, b=b1: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "2-D SPD quadratic with closed-form minimizer",
        },
        {
            "id": "D2",
            "name": "ill-conditioned quadratic",
            "x0": np.array([1.8, -1.5]),
            "loss": lambda x, A=A2, b=b2: quadratic_loss(A, b, x),
            "grad": lambda x, A=A2, b=b2: quadratic_grad(A, b, x),
            "project": lambda x: x,
            "dim": 2,
            "info": "anisotropic bowl with coupled coordinates",
        },
        {
            "id": "D3",
            "name": "nonconvex Rosenbrock-ripple",
            "x0": np.array([-1.2, 1.0]),
            "loss": rosenbrock_loss,
            "grad": rosenbrock_grad,
            "project": lambda x: x,
            "dim": 2,
            "info": "curved valley plus small multimodal ripple",
        },
        {
            "id": "D4",
            "name": "real logistic loss",
            "x0": np.zeros(2),
            "loss": lambda w, X=X2, y=y2: logistic_loss(w, X, y, 0.02),
            "grad": lambda w, X=X2, y=y2: logistic_grad(w, X, y, 0.02),
            "project": lambda x: x,
            "dim": 2,
            "X": X2,
            "y": y2,
            "info": "NumPy logistic regression objective on a fixed small dataset",
        },
        {
            "id": "D5",
            "name": "high-dimensional sparse constrained case",
            "x0": np.zeros(d),
            "loss": lambda w, X=Xh, y=yh: l1_logistic_objective(w, X, y, 0.04),
            "grad": lambda w, X=Xh, y=yh: logistic_grad(w, X, y, 0.0),
            "project": lambda x: project_l2_ball(x, 2.5),
            "dim": d,
            "X": Xh,
            "y": yh,
            "A": A5,
            "b": b5,
            "info": "32-D sparse logistic objective with L1 and norm constraint",
        },
    ]


def preview_ladder(ladder):
    for rung in ladder:
        sample = rung["x0"][: min(5, rung["dim"])]
        print(rung["id"], rung["name"], "dim=", rung["dim"], "sample=", np.round(sample, 3), "--", rung["info"])


def contour_values(rung, grid=80, span=2.4):
    xs = np.linspace(-span, span, grid)
    ys = np.linspace(-span, span, grid)
    Z = np.zeros((grid, grid))
    base = rung["x0"].astype(float).copy()
    for i, xval in enumerate(xs):
        for j, yval in enumerate(ys):
            probe = base.copy()
            probe[0] = xval
            probe[1] = yval
            Z[j, i] = rung["loss"](probe)
    return xs, ys, Z


def plot_trajectory_summary(results, metric_label="final loss"):
    fig, axes = plt.subplots(2, 5, figsize=(18, 6))
    for col, item in enumerate(results):
        rung = item["rung"]
        path = item["path"]
        losses = item["losses"]
        xs, ys, Z = contour_values(rung)
        axes[0, col].contour(xs, ys, Z, levels=18, cmap="viridis")
        axes[0, col].plot(path[:, 0], path[:, 1], marker="o", markersize=2, linewidth=1)
        axes[0, col].set_title(rung["id"])
        axes[1, col].plot(losses)
        axes[1, col].set_title(metric_label)
        axes[1, col].set_xlabel("iteration")
    plt.tight_layout()


def print_metric_table(results, metric_label="final loss"):
    print(f"{'rung':<4} {'name':<38} {'iters':>6} {metric_label:>14}")
    for item in results:
        final_value = item["metric"]
        iters = len(item["losses"]) - 1
        print(f"{item['rung']['id']:<4} {item['rung']['name']:<38} {iters:>6} {final_value:>14.6f}")

## The concept, built once (D1)

A subgradient satisfies $f(z)\ge f(x)+g^T(z-x)$, and the L1 prox is $S_\lambda(v)=\mathrm{sign}(v)\max(|v|-\lambda,0)$.

Check the absolute-value support line and the soft-threshold vector from the lesson. These asserts lock the corner calculus to the exact worked numbers.

In [ ]:
def support_line_value(x, g, z):
    return abs(x) + g * (z - x)


def soft_threshold_prox(v, lam):
    return np.sign(v) * np.maximum(np.abs(v) - lam, 0.0)


g = 0.5
line_at_two = support_line_value(0.0, g, 2.0)
line_at_neg_two = support_line_value(0.0, g, -2.0)
prox_vector = soft_threshold_prox(np.array([-2.0, -0.5, 0.5, 2.0]), 1.0)

assert np.isclose(line_at_two, 1.0)
assert np.isclose(abs(2.0), 2.0)
assert np.isclose(line_at_neg_two, -1.0)
assert np.allclose(prox_vector, np.array([-1.0, 0.0, 0.0, 1.0]))
print("support at z=2", line_at_two)
print("prox", prox_vector)

Build proximal subgradient descent. The smooth gradient moves first; then the prox uses the scaled threshold $\eta\lambda$.

In [ ]:
def proximal_subgradient_descent(rung, steps=100, eta0=0.18, lam=0.04):
    x = rung["x0"].astype(float).copy()
    path = [x.copy()]
    losses = [rung["loss"](x)]
    for t in range(steps):
        eta = eta0 / math.sqrt(t + 1.0)
        x = x - eta * rung["grad"](x)
        x = soft_threshold(x, eta * lam)
        x = rung["project"](x)
        path.append(x.copy())
        losses.append(rung["loss"](x))
    return np.array(path), np.array(losses)

## The dataset ladder

Family F4 uses the same inline D1-D5 ladder: quadratic, ill-conditioned, nonconvex, real logistic loss, and high-dimensional sparse constrained loss.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    ladder = make_loss_surface_ladder()
    preview_ladder(ladder)
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Run the same method across D1-D5

The metric is the plan's requested final loss, final objective, feasible loss, or primal-dual gap.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    ladder = make_loss_surface_ladder()
    results = []
    for rung in ladder:
        path, losses = proximal_subgradient_descent(rung, steps=120, eta0=0.16, lam=0.04)
        results.append({"rung": rung, "path": path, "losses": losses, "metric": losses[-1]})
    print_metric_table(results, "final objective")
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Results visualization

The closing figure has trajectory-on-contours panels and a summary metric curve.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    plot_trajectory_summary(results, "objective curve")
    metrics = [item["metric"] for item in results]
    plt.figure(figsize=(6, 3))
    plt.plot([item["rung"]["id"] for item in results], metrics, marker="o")
    plt.ylabel("final objective")
    plt.title("Proximal/subgradient objective by rung")
    plt.grid(True, alpha=0.3)
    plt.show()
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on the hardest rung

Pitfall on D5: forgetting the prox scale. In proximal gradient, the threshold is $\eta\lambda$, not bare $\lambda$.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    def wrong_unscaled_prox(rung, steps=40, eta=0.12, lam=0.04):
        x = rung["x0"].astype(float).copy()
        losses = []
        for _ in range(steps):
            x = x - eta * rung["grad"](x)
            x = soft_threshold(x, lam)
            x = rung["project"](x)
            losses.append(rung["loss"](x))
        return np.array(losses), x


    def fixed_scaled_prox(rung, steps=40, eta=0.12, lam=0.04):
        x = rung["x0"].astype(float).copy()
        losses = []
        for t in range(steps):
            rate = eta / math.sqrt(t + 1.0)
            x = x - rate * rung["grad"](x)
            x = soft_threshold(x, rate * lam)
            x = rung["project"](x)
            losses.append(rung["loss"](x))
        return np.array(losses), x


    hard = ladder[-1]
    wrong_losses, wrong_x = wrong_unscaled_prox(hard)
    fixed_losses, fixed_x = fixed_scaled_prox(hard)
    print("wrong threshold final", wrong_losses[-1], "nonzeros", np.count_nonzero(wrong_x))
    print("scaled threshold final", fixed_losses[-1], "nonzeros", np.count_nonzero(fixed_x))
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Evaluate it + Practice

- Metric: compare the final value to a no-skill baseline that returns the initial point.
- Sanity check: on D1, verify the path moves toward the closed-form quadratic solution or the asserted lesson numbers.
- Ablation: turn off the key idea, such as newest-value updates, the prox, projection, dual lower bound, uniform sampling, or PSD check.
- Failure signals: rising loss, exploding iterates, negative inequality multipliers, invalid lower bounds, or dense non-sparse D5 solutions.
- CPU note: these examples are seeded, small, and NumPy-only; do not execute the notebook as part of rebuilding.

Practice: Try $g=-0.5$ at the absolute-value corner and plot the changed path.

Practice: Vary $\lambda$ and record D5 sparsity versus objective.

Practice: Swap the L1 prox for a box projection and compare the constraint interpretation.